In [ ]:
#!pip install --user huggingface_hub requests python-dotenv pandas numpy

In [ ]:
import json
import re
from dataclasses import dataclass, field, asdict
from typing import Any, Optional
from datetime import datetime
import logging

from huggingface_hub import InferenceClient

import time
from functools import wraps

In [ ]:
client = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct",
    token="[YOUR TOKEN HERE]",
)

# Comment above block and uncomment bottom block if running from VM and not Colab

# from dotenv import load_dotenv
# import os
# from pathlib import Path

# env_path = Path.cwd()

# load_dotenv(env_path / ".env")

# client = InferenceClient(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     token=os.getenv("HF_TOKEN"),
# )



In [20]:
logging.basicConfig(
    level  = logging.INFO,
    format = "%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt= "%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("synthetic_pipeline")

In [3]:
# Define Constants

MODEL_ID          = "Qwen/Qwen2.5-7B-Instruct"
VALID_SEVERITIES  = {"minor", "moderate", "severe"}
REQUIRED_LABEL_KEYS = {"damage_type", "severity", "vehicle_part"}

In [4]:
# Define Custom Exceptions

# Output cannot be parsed as JSON
class ParseError(Exception):
    """Raised when Qwen output cannot be parsed as JSON."""

# Output structure is wrong
class ValidationError(Exception):
    """Raised when parsed JSON fails schema validation."""

# All retries have been exhausted without successful generation
class GenerationError(Exception):
    """Raised when all retry attempts are exhausted."""

In [ ]:
def with_retry(max_retries: int = 3, base_delay: float = 2.0, backoff: float = 2.0):
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            delay = base_delay
            for attempt in range(1, max_retries + 1):
                try:
                    return fn(*args, **kwargs)

                except (ParseError, ValidationError) as e:
                    log.warning(
                        f"[{attempt}/{max_retries}] {type(e).__name__} — {e}"
                    )
                    if attempt == max_retries:
                        log.error("Max retries reached. Raising GenerationError.")
                        raise GenerationError(
                            f"All {max_retries} attempts failed: {e}"
                        ) from e

                except Exception as e:
                    log.warning(
                        f"[{attempt}/{max_retries}] Unexpected error — {e}"
                    )
                    if attempt == max_retries:
                        log.error("Max retries reached. Raising GenerationError.")
                        raise GenerationError(
                            f"All {max_retries} attempts failed: {e}"
                        ) from e

                log.info(f"Retrying in {delay:.1f}s ...")
                time.sleep(delay)
                delay *= backoff

        return wrapper
    return decorator


print("Logging and retry decorator ready")

✅ Logging and retry decorator ready


In [5]:
# Input Schema dataclass

@dataclass
class PromptRequest:
    data_type:      str
    description:    str
    format_spec:    dict[str, Any]
    quantity:       int
    domain_context: str

    def to_dict(self) -> dict:
        return {
            "data_type":      self.data_type,
            "description":    self.description,
            "format_spec":    self.format_spec,
            "quantity":       self.quantity,
            "domain_context": self.domain_context,
        }

In [ ]:
@dataclass
class DamageLabel:
    damage_type:  str
    severity:     str
    vehicle_part: str

@dataclass
class ReferenceRecord:
    image_id:       str
    labels:         DamageLabel
    domain:         str
    source:         str
    created_at:     str
    metadata:       dict

    def to_dict(self) -> dict:
        d = asdict(self)
        d["labels"] = asdict(self.labels)
        return d

    def to_qdrant_payload(self) -> dict:
        return {
            "image_id":    self.image_id,
            "damage_type": self.labels.damage_type,
            "severity":    self.labels.severity,
            "vehicle_part":self.labels.vehicle_part,
            "domain":      self.domain,
            "source":      self.source,
            "created_at":  self.created_at,
            **self.metadata,
        }

In [6]:
# Converts a PrompRequest into a two-message list that Qwen expects

def build_prompt(
    req: PromptRequest,
    retrieved_context: Optional[str] = None,
) -> list[dict]:
    fields_block = "\n".join(
        f"  - {f}" for f in req.format_spec.get("required_fields", [])
    )

    constraints = req.format_spec.get("constraints", {})
    constraints_block = ""
    if constraints:
        constraints_block = "\nAdditional constraints:\n" + "\n".join(
            f"  - {k}: {v}" for k, v in constraints.items()
        )

    rag_block = ""
    if retrieved_context:
        rag_block = (
            "\nRetrieved reference examples (use these to ground your output):\n"
            "---\n"
            f"{retrieved_context.strip()}\n"
            "---\n"
        )

    system_prompt = (
        f"You are a synthetic data generation assistant specializing in {req.domain_context}.\n"
        "You output ONLY valid JSON. No explanation, no markdown, no code fences.\n"
        "Your response must be parseable by json.loads() directly."
    )

    user_prompt = (
        f"Task: {req.description}\n\n"
        f"Data type     : {req.data_type}\n"
        f"Domain        : {req.domain_context}\n"
        f"Quantity      : {req.quantity} records\n\n"
        f"Required fields per record:\n{fields_block}"
        f"{constraints_block}\n"
        f"{rag_block}\n"
        f"Return EXACTLY this JSON envelope — nothing outside it:\n"
        f'{{\n'
        f'  "data_type": "{req.data_type}",\n'
        f'  "records": [\n'
        f'    {{\n'
        f'      "image_id": "img_001",\n'
        f'      "labels": {{\n'
        f'        "damage_type": "string",\n'
        f'        "severity": "minor|moderate|severe",\n'
        f'        "vehicle_part": "string"\n'
        f'      }}\n'
        f'    }}\n'
        f'  ]\n'
        f'}}\n\n'
        f"Generate {req.quantity} unique, realistic records. "
        f"Start your response with {{ and end with }}."
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]

In [7]:
# Actual API call to Qwen with retry logic

def call_qwen(
    messages:   list[dict],
    client:     InferenceClient,
    max_tokens: int = 800,
) -> str:
    response = client.chat_completion(messages=messages, max_tokens=max_tokens)
    return response.choices[0].message.content

In [8]:
# Takes Qwen's raw output and extracts the JSON object.

def parse_output(raw: str) -> dict:
    text = raw.strip()

    if text.startswith("```"):
        text = re.sub(r"^```[a-z]*\n?", "", text)
        text = re.sub(r"\n?```$",       "", text.strip())
        text = text.strip()

    start = text.find("{")
    end   = text.rfind("}")
    if start == -1 or end == -1:
        raise ParseError(
            f"No JSON object found in model output.\nRaw (first 300 chars): {raw[:300]}"
        )
    text = text[start : end + 1]

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ParseError(
            f"json.loads() failed: {e}\nCleaned text (first 300 chars): {text[:300]}"
        )

In [9]:
# Helper function to check whether value is Null or emptry string.
# If so, appends error message to errors list and returns False, Otherwise returns True.

def _assert_not_null(value: Any, path: str, errors: list[str]) -> bool:
    if value is None or (isinstance(value, str) and value.strip() == ""):
        errors.append(f"Null/empty value at '{path}'")
        return False
    return True

In [10]:
# Runs all validation checks on the parsed JSON and returns list of records if valid

def validate_records(data: dict) -> list[dict]:
    errors: list[str] = []

    if "records" not in data:
        raise ValidationError("Missing top-level 'records' key.")
    if not isinstance(data["records"], list) or len(data["records"]) == 0:
        raise ValidationError("'records' must be a non-empty list.")

    records   = data["records"]
    seen_ids: dict[str, int] = {}

    for i, record in enumerate(records):
        p = f"Record[{i}]"

        img_id = None
        if "image_id" not in record:
            errors.append(f"{p}: missing 'image_id'")
        else:
            img_id = record["image_id"]
            if not _assert_not_null(img_id, f"{p}.image_id", errors):
                img_id = None

        if "labels" not in record:
            errors.append(f"{p}: missing 'labels'")
            continue
        if not isinstance(record["labels"], dict):
            errors.append(f"{p}.labels: must be a dict, got {type(record['labels']).__name__}")
            continue

        labels = record["labels"]

        for key in REQUIRED_LABEL_KEYS:
            if key not in labels:
                errors.append(f"{p}.labels: missing required key '{key}'")
            else:
                _assert_not_null(labels[key], f"{p}.labels.{key}", errors)

        if "severity" in labels and labels["severity"] not in VALID_SEVERITIES:
            errors.append(
                f"{p}.labels.severity: '{labels['severity']}' "
                f"not in {VALID_SEVERITIES}"
            )

        if img_id is not None:
            if img_id in seen_ids:
                errors.append(
                    f"{p}: duplicate image_id '{img_id}' "
                    f"(first seen at Record[{seen_ids[img_id]}])"
                )
            else:
                seen_ids[img_id] = i

    if errors:
        raise ValidationError(
            f"{len(errors)} validation error(s):\n" +
            "\n".join(f"  • {e}" for e in errors)
        )

    return records

In [23]:
# Main function that orchestrates the entire generation process with retries

@with_retry(max_retries=3, base_delay=2.0, backoff=2.0)
def generate_synthetic_data(
    prompt_request:    PromptRequest,
    client:            InferenceClient,
    retrieved_context: Optional[str] = None,
    max_tokens:        int = 800,
    max_retries:       int = 3,
) -> dict:
    log.info(
        f"Generating {prompt_request.quantity}× "
        f"{prompt_request.data_type} | "
        f"domain: {prompt_request.domain_context} | "
        f"RAG: {'yes' if retrieved_context else 'no'}"
    )

    messages = build_prompt(prompt_request, retrieved_context)
    raw      = call_qwen(messages, client, max_tokens)
    parsed   = parse_output(raw)
    records  = validate_records(parsed)

    actual = len(records)
    if actual != prompt_request.quantity:
        log.warning(
            f"Quantity mismatch: wanted {prompt_request.quantity}, got {actual}"
        )

    log.info(f"{actual} record(s) validated successfully")
    return parsed


print("generate_synthetic_data() ready")


print("All 5 functions ready")

generate_synthetic_data() ready
All 5 functions ready


In [ ]:
def normalize_to_reference(
    generated: dict,
    domain:    str = "auto insurance claims",
    source:    str = "synthetic",
    metadata:  Optional[dict] = None,
) -> list[ReferenceRecord]:
    records   = []
    timestamp = datetime.utcnow().isoformat()

    for raw in generated["records"]:
        record = ReferenceRecord(
            image_id   = raw["image_id"],
            labels     = DamageLabel(
                damage_type  = raw["labels"]["damage_type"],
                severity     = raw["labels"]["severity"],
                vehicle_part = raw["labels"]["vehicle_part"],
            ),
            domain     = domain,
            source     = source,
            created_at = timestamp,
            metadata   = metadata or {},
        )
        records.append(record)

    return records

In [25]:
# TEST BLOCK
# Run this cell to test each function individually with assertions and print statements
# Adjust the PromptRequest parameters as needed for different scenarios

req = PromptRequest(
    data_type       = "image_metadata",
    description     = "Generate metadata for damaged vehicle claim images",
    format_spec     = {
        "required_fields": ["image_id", "labels.damage_type",
                            "labels.severity", "labels.vehicle_part"],
        "constraints":     {"severity":  "minor | moderate | severe",
                            "image_id":  "format img_001, img_002, ..."},
    },
    quantity        = 5,
    domain_context  = "auto insurance claims",
)

messages = build_prompt(req)
assert len(messages) == 2 and messages[0]["role"] == "system"
print("build_prompt()")

messages_rag = build_prompt(req, retrieved_context="Example: img_000, dent, minor, hood")
assert "Retrieved reference examples" in messages_rag[1]["content"]
print("build_prompt(retrieved_context=...)")

raw = call_qwen(messages, client)
assert isinstance(raw, str) and len(raw) > 0
print("call_qwen()")

parsed = parse_output(raw)
assert "records" in parsed
print("parse_output()")

records = validate_records(parsed)
assert isinstance(records, list)
print("validate_records()")

result = generate_synthetic_data(req, client)
assert len(result["records"]) > 0
print("generate_synthetic_data()")

print("\n── Final output ─────────────────────────────────────────────")
print(json.dumps(result, indent=2))

sample = {
    "data_type": "image_metadata",
    "records": [
        {"image_id": "img_001", "labels": {"damage_type": "dent",  "severity": "minor",    "vehicle_part": "hood"}},
        {"image_id": "img_002", "labels": {"damage_type": "crack", "severity": "moderate", "vehicle_part": "windshield"}},
    ]
}

reference_records = normalize_to_reference(sample, metadata={"model_version": "Qwen2.5-7B"})

print("── ReferenceRecord (structured) ─────────────────────────────")
for r in reference_records:
    print(json.dumps(r.to_dict(), indent=2))

print("\n── Qdrant payload (flat) ────────────────────────────────────")
for r in reference_records:
    print(json.dumps(r.to_qdrant_payload(), indent=2))

print(f"{len(reference_records)} reference record(s) normalized")


build_prompt()
build_prompt(retrieved_context=...)


2026-03-13 15:39:57 | INFO     | Generating 5× image_metadata | domain: auto insurance claims | RAG: no


call_qwen()
parse_output()
validate_records()


2026-03-13 15:39:59 | INFO     | 5 record(s) validated successfully


generate_synthetic_data()

── Final output ─────────────────────────────────────────────
{
  "data_type": "image_metadata",
  "records": [
    {
      "image_id": "img_001",
      "labels": {
        "damage_type": "fender dent",
        "severity": "minor",
        "vehicle_part": "fender"
      }
    },
    {
      "image_id": "img_002",
      "labels": {
        "damage_type": "cracked windshield",
        "severity": "moderate",
        "vehicle_part": "windshield"
      }
    },
    {
      "image_id": "img_003",
      "labels": {
        "damage_type": "bent door panel",
        "severity": "severe",
        "vehicle_part": "door"
      }
    },
    {
      "image_id": "img_004",
      "labels": {
        "damage_type": "chipped bumper",
        "severity": "minor",
        "vehicle_part": "bumper"
      }
    },
    {
      "image_id": "img_005",
      "labels": {
        "damage_type": "ripped seat fabric",
        "severity": "moderate",
        "vehicle_part": "seat"
      }


In [26]:
# ── Mock RAG Test ─────────────────────────────────────────────────────────────
# Simulates what Qdrant will return — hardcoded retrieval results.
# Once Qdrant is ready, replace mock_retrieved_context with actual retrieval output.

mock_retrieved_context = """
Example 1:
{"image_id": "img_091", "labels": {"damage_type": "rear-end collision", "severity": "severe", "vehicle_part": "rear bumper"}}

Example 2:
{"image_id": "img_092", "labels": {"damage_type": "hail damage", "severity": "minor", "vehicle_part": "hood"}}

Example 3:
{"image_id": "img_093", "labels": {"damage_type": "side swipe", "severity": "moderate", "vehicle_part": "driver door"}}
"""

req_rag = PromptRequest(
    data_type      = "image_metadata",
    description    = "Generate metadata for damaged vehicle claim images",
    format_spec    = {
        "required_fields": ["image_id", "labels.damage_type",
                            "labels.severity", "labels.vehicle_part"],
        "constraints":     {"severity": "minor | moderate | severe",
                            "image_id": "format img_001, img_002, ..."},
    },
    quantity       = 5,
    domain_context = "auto insurance claims",
)

print("── Without RAG context ──────────────────────────────────────")
result_no_rag = generate_synthetic_data(req_rag, client)

print("\n── With mock RAG context ────────────────────────────────────")
result_with_rag = generate_synthetic_data(req_rag, client, retrieved_context=mock_retrieved_context)

print("\n── Comparison ───────────────────────────────────────────────")
print("Without RAG:")
for r in result_no_rag["records"]:
    print(f"  {r['image_id']} | {r['labels']['damage_type']} | {r['labels']['severity']}")

print("\nWith RAG:")
for r in result_with_rag["records"]:
    print(f"  {r['image_id']} | {r['labels']['damage_type']} | {r['labels']['severity']}")

2026-03-13 15:40:09 | INFO     | Generating 5× image_metadata | domain: auto insurance claims | RAG: no


── Without RAG context ──────────────────────────────────────


2026-03-13 15:40:11 | INFO     | 5 record(s) validated successfully
2026-03-13 15:40:11 | INFO     | Generating 5× image_metadata | domain: auto insurance claims | RAG: yes



── With mock RAG context ────────────────────────────────────


2026-03-13 15:40:14 | INFO     | 5 record(s) validated successfully



── Comparison ───────────────────────────────────────────────
Without RAG:
  img_001 | fender dent | minor
  img_002 | cracked windshield | moderate
  img_003 | door scratch | minor
  img_004 | bent bumper | severe
  img_005 | chipped paint | moderate

With RAG:
  img_001 | fender bender | minor
  img_002 | tree branch impact | moderate
  img_003 | rollover | severe
  img_004 | pothole impact | minor
  img_005 | parking lot scratch | minor
